# Implementing Canonicalizing Open Knowledge Bases 

### Paper Title: Canonicalizing Open Knowledge Bases
Authors: Luis Galárraga, Geremy Heitz, Kevin Murphy, and Fabian Suchanek (CIKM 2014)

## Motivation and Problem Statement

### The Problem
Open Information Extraction (Open IE) systems like ReVerb or NELL extract facts from web text as triples:
$⟨subject,predicate,object⟩$<br>
For example:
- $⟨ Obama, was born in, Honolulu⟩$
- $⟨ Barack Obama, place of birth, Honolulu⟩$

But as we know:
- Obama and Barack Obama are the same person as well as
- was born in and place of birth are same relation.

These systems as we can see do not canonicalize names or relations which leads to:
- Redundancy
- Ambiguity, and
- Incomplete or polluted knowledge queries.




### Goal:
Canonicalize entities and relations in large Open IE knowledge bases by:
- Clustering synonymous entity mentions (e.g., "Obama", "President Obama")
- Clustering semantically equivalent relations (e.g., "was born in", "birthplace is")

This bridges the gap between:
- Open IE: High recall, low precision
- Closed IE: High precision, low coverage

## Assumptions, Goals, and Research Questions

### Assumptions:
- Mentions in the same web page refer to the same entity.
- Clustering can recover real-world canonical forms from noisy extractions.
- Features like string similarity, attribute overlap, and type constraints help identify synonymy.

### Main Research Questions:
- Can we cluster noun phrases into canonical entity groups?
- Can we cluster relation phrases into semantically coherent groups?
- What features and design choices lead to the most effective clustering?

## Methodology

### **Overview of Pipeline:**

#### A. Entity Canonicalization

* **Input**: Triples from ReVerb or NELL:
  $\langle \text{subject}, \text{relation}, \text{object} \rangle$
* **Step 1**: Create “mentions” (noun phrases with context)
* **Step 2**: Cluster mentions using Hierarchical Agglomerative Clustering (HAC)
* **Step 3**: Use blocking (canopies) to improve HAC efficiency
* **Step 4**: Define various similarity functions between mentions
* **Step 5**: Learn a weighted similarity metric using logistic regression


#### B. Relation Canonicalization
* **Step 1**: Use AMIE (a rule mining system) to find **equivalent relations**
* **Step 2**: Convert subsumption rules into **clusters of equivalent phrases**
* **Step 3**: Optionally map clusters to **Freebase relations**


### A. **Mathematical Models: Entity Similarity and Clustering**

Each **mention** is:

$$
m = (n, u, A)
$$

Where:

* $n$: noun phrase (e.g., "President Obama")
* $u$: web URL
* $A$: attributes (i.e., $(\text{predicate}, \text{object})$ pairs)

#### Similarity Functions:

All operate on mention pairs $m, m'$.

* **String Identity**:

  $$
  f_{\text{strid}}(m, m') = 
  \begin{cases}
  1 & \text{if } n = n' \\
  0 & \text{otherwise}
  \end{cases}
  $$

* **Jaro-Winkler Similarity**: For fuzzy string matching

  $$
  f_{\text{strsim}}(m, m') = \text{JaroWinkler}(n, n')
  $$

* **Attribute Overlap** (Jaccard over (predicate, object)):

  $$
  f_{\text{attr}}(m, m') = \frac{|A \cap A'|}{|A \cup A'|}
  $$

* **IDF Token Overlap** (weighted word overlap):

  $$
  f_{\text{itok}}(m, m') = \frac{\sum_{w \in w(n) \cap w(n')} \text{IDF}(w)}{\sum_{w \in w(n) \cup w(n')} \text{IDF}(w)}
  $$

* **Word Overlap (page text)**: Jaccard over TF-IDF top-100 page words

* **Entity Overlap**: Jaccard over linked Freebase entities per page

* **Type Overlap**: Jaccard over inferred entity types



#### Combined Similarity:

$$
f_{\text{sim}}(m, m') = \sigma\left(c_0 + \sum_i c_i f_i(m, m')\right)
$$

Where:

* $\sigma$: Logistic function
* $f_i$: each feature described above
* $c_i$: learned coefficients via logistic regression


### Clustering Algorithm:

**Hierarchical Agglomerative Clustering (HAC)** using **single linkage**

* Merge clusters that have **maximum similarity**
* Use **token-based blocking (canopies)** to reduce pairwise comparisons


### Canonicalization:

Once a cluster is formed, choose the **canonical name** as:

* The noun phrase that appears in the **most distinct sources**
* If tie: choose the **longest phrase**


### B. **Relation Canonicalization via Rule Mining (AMIE)**

#### Step 1: Semi-canonicalize the KB

Subjects and objects are canonicalized, but predicates are still raw phrases.

#### Step 2: Use AMIE to find **subsumption rules**:

$$
r(x, y) \Rightarrow r'(x, y)
$$

If both directions exist: infer **equivalence**:

$$
r(x, y) \Leftrightarrow r'(x, y)
$$

Rules scored by **PCA Confidence**:

$$
\text{conf}_{\text{PCA}} = \frac{\text{support of rule}}{\text{support under PCA assumption}}
$$

This helps deal with KB **incompleteness**.

#### Step 3: Cluster relation phrases using transitive closure

#### Step 4: Map relation clusters to **Freebase schema** using:

* Matching patterns
* Co-occurrence
* Rule mining

## Comparison to Prior Work

| Method           | Entity Canonicalization | Relation Canonicalization | Notes                                         |
| ---------------- | ----------------------- | ------------------------- | --------------------------------------------- |
| Resolver         | Yes                     | Yes                       | Uses probabilistic modeling                   |
| Concept Resolver | Yes                     | No                        | Relies on NELL ontology                       |
| WEBRE            | Yes                     | Yes                       | Uses typed verb patterns                      |
| **This Paper**   | Yes                     | Yes                       | Combines OpenIE with rule mining and blocking |


## Experimental Setup

* **Corpus**: ReVerb triples over ClueWeb09 (millions of triples)
* **Gold standard**: Freebase-linked mentions and human evaluation
* **Two datasets**:

  * **Base**: clean entity data
  * **Ambiguous**: includes homonyms and polysemous names

### Evaluation Metrics:

Three levels for precision, recall, and F1:

* **Macro**: How many full clusters are pure
* **Micro**: Based on dominant entity per cluster
* **Pairwise**: Based on mention-pairs' correctness


## Results and Analysis

### Entity Clustering (on ReVerb):

| Feature           | Macro F1 | Micro F1 | Pairwise F1 |
| ----------------- | -------- | -------- | ----------- |
| String identity   | 0.61     | 0.89     | 0.85        |
| IDF token overlap | 0.93     | 0.98     | 0.99        |
| Attribute overlap | 0.10     | 0.37     | 0.17        |
| Type overlap      | 0.96     | 0.98     | 0.99        |
| **Simple ML**     | **0.94** | **0.98** | **0.99**    |

Takeaway: **IDF token overlap and type overlap are strongest individual features**


### Relation Clustering (on ReVerb):

| KB Type           | Conf. | Phrases | Clusters | Macro P | Micro P | Pairwise P |
| ----------------- | ----- | ------- | -------- | ------- | ------- | ---------- |
| Linked KB         | 0.8   | 522     | 118      | 0.90    | 0.94    | 0.95       |
| Linked KB + types | 0.8   | 752     | 303      | 0.95    | 0.98    | 0.997      |

Takeaway: **Adding type constraints improves relation clustering precision**


## Key Contributions

* A **practical, scalable method** for entity and relation canonicalization in OpenIE KBs
* Demonstrated that **simple ML features** + **blocking + rule mining** yield high-precision clusters
* Bridged **open** and **closed** IE systems with a hybrid model


## Limitations

* **Sparse attribute data** limits attribute overlap features
* **Ambiguous names** can reduce clustering precision
* Rule mining only covers a **subset** of relation phrases


## Summary

This paper proposes an efficient pipeline for:

* Clustering entity mentions into canonical groups
* Clustering synonymous relation phrases via rule mining
* Mapping canonical forms to a schema (e.g., Freebase)

The system is **modular**, scalable, and produces **high-precision clusters**—a strong step toward structuring Open IE data with minimal supervision.
